# PyTorch Day Korea 2026 · NVIDIA PhysicsNeMo 실습

2026년 11월 21일 · 강사 양민규(NVIDIA)

양끝을 고정한 줄의 처짐을 PINN으로 구하고, 알루미늄 판 위의 히터를 옮겨 가며 FNO로 온도 분포를 예측합니다.

| 실습 | 풀어 볼 문제 |
| --- | --- |
| [1.1 · 줄의 처짐](01_PINN.ipynb#lab-11) | 하중이 일정할 때 줄의 가운데는 얼마나 내려갈까요? |
| [1.2 · 줄의 길이](01_PINN.ipynb#lab-12) | 같은 하중을 받는 줄의 길이를 두 배로 늘리면 처짐은 몇 배가 될까요? |
| [1.3 · 하중 추정](01_PINN.ipynb#lab-13) | 줄 위 100개 지점의 높이로 위치별 하중을 알아낼 수 있을까요? |
| [2 · 열판의 온도](02_FNO.ipynb) | 같은 2 W 히터를 중앙과 모서리 근처에 놓으면 어느 쪽의 최고 온도가 높을까요? |

**JupyterLab을 열었다면** 아래 1절부터 시작하세요. GitHub에서는 노트북을 읽을 수 있습니다. 코드를 실행하려면 CUDA GPU가 있는 [실습 환경](environment/README.md)이 필요합니다.

현장에서는 행사에서 안내받은 링크로 Brev에 로그인한 뒤 `Deploy Launchable`을 한 번 누르세요. [Brev 콘솔](https://brev.nvidia.com/)에서 내 인스턴스가 `Running`이면 `JupyterLab`을 엽니다. 파일 목록이 보이면 교재 폴더에서 `00_Start_Here.ipynb`를 여세요.


<a id="environment-check"></a>

## 1. 커널과 GPU 확인

JupyterLab 오른쪽 위에서 `AI4Sci PhysicsNeMo 2.2.2 (uv / CUDA)` 커널을 선택하세요. 코드 셀에 커서를 두고 `Shift + Enter`를 누르면 실행됩니다. 첫 셀은 교재 경로를, 두 번째 셀은 패키지 버전과 GPU 이름을 표시합니다.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import importlib.metadata as meta
import platform

expected = {
    "nvidia-physicsnemo": "2.2.2", "torch": "2.10.0+cu128",
    "numpy": "2.5.2", "sympy": "1.14.0", "PyYAML": "6.0.3",
    "h5py": "3.16.0", "matplotlib": "3.11.2",
}
print("Python:", platform.python_version(), "(실습 환경 3.12)")
missing = []
for package, target in expected.items():
    try:
        actual = meta.version(package)
        print(f"{package}: {actual}" + (" ✓" if actual == target else f" (실습 버전 {target})"))
    except meta.PackageNotFoundError:
        missing.append(package)
if missing:
    raise RuntimeError("실습 커널을 선택하세요. 설치되지 않은 패키지: " + ", ".join(missing))

import torch
from physicsnemo.models.mlp import FullyConnected
from physicsnemo.models.fno import FNO
from physicsnemo.sym.eq.phy_informer import PhysicsInformer
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU를 사용할 수 없습니다. 실습 커널과 인스턴스의 GPU 설정을 확인하세요.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime:", torch.version.cuda)
x = torch.tensor([1.0], device="cuda", requires_grad=True)
(x*x).sum().backward()
assert torch.isfinite(x.grad).all()
print("GPU tensor와 미분 확인 완료")


## 2. 열판 데이터 확인

한 사례에는 8×8 mm 히터가 단위 면적당 공급하는 전력(W/m²)과 그때의 정상상태 온도(°C)가 들어 있습니다. 두 값 모두 64×64 격자에 저장합니다. 온도는 열전도 방정식을 풀어 구한 값입니다.

아래 셀을 실행해 학습 8,000개, 검증 1,000개, 시험 1,000개가 표시되는지 확인하세요. 데이터 폴더가 없는 환경에서는 처음 실행할 때 생성합니다.


In [ ]:
import h5py
from brev.install import prepare_data

DATA_DIR = ROOT / "data/heated_plate"
files = [DATA_DIR / f"{name}.hdf5" for name in ("train", "val", "test")]
if not all(path.is_file() for path in files):
    prepare_data(ROOT, Path(sys.prefix))
for name, path in zip(("train", "validation", "test"), files):
    with h5py.File(path, "r") as data:
        if data.attrs.get("schema") != "insulated_aluminum_plate_heater_v1":
            raise RuntimeError("히터 데이터가 아닙니다. data/heated_plate 폴더를 확인하세요.")
        print(f"{name}: 발열량 {data['f'].shape} W/m², 판 온도 {data['u'].shape} °C")
print("데이터 준비 완료:", DATA_DIR)


## 다음 실습

GPU와 데이터를 확인했다면 [Lab 1.1 · 양끝이 고정된 줄의 처짐](01_PINN.ipynb#lab-11)을 여세요.

### 더 읽어 보기

- [PhysicsNeMo 기초 개념](01_Introduction.ipynb)
- [PINN의 식과 학습 코드](01_labs/01_pinn/Lab_1_PINN_Fundamentals.ipynb)
- [포물선 운동](01_labs/02_projectile/Lab_2_Projectile_Motion.ipynb)
- [열전도](01_labs/03_heat_conduction/Lab_3_Heat_Conduction.ipynb)
- [FourCastNet으로 날씨 예측하기](01_labs/04_weather_forecasting/Lab_4_Weather_Forecasting.ipynb)
